In [ ]:
##########################################
#####  Phase 5.1. Scoring genotypes  #####
##########################################

import pandas as pd
import numpy as np
import scipy.stats as stats
import os

base_dir = './'
phase4_path = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')
phase3_path = os.path.join(base_dir, 'Phase3_BLUP_Indices.csv')

df_p4 = pd.read_csv(phase4_path)
df_p3 = pd.read_csv(phase3_path)

# Trait selection (Treatment p < 0.05 AND Var_GxE >= 15.0)
top_traits_df = df_p4[(df_p4['Treatment_p_value'] < 0.05) & (df_p4['Var_GxE(%)'] >= 15.0)].copy()
selected_traits = top_traits_df['Trait'].tolist()

scoring_data = df_p3[df_p3['Trait'].isin(selected_traits)].copy()

print(f"Scoring {len(selected_traits)} selected traits...")

# Set the determination metric of traits (for particular traits use either DRI or STI)
def determine_metric(trait_name):
    # For structure data, apply DRI 
    dri_keywords = ['structure_trait'] 
    if any(keyword in trait_name for keyword in dri_keywords):
        return 'BLUP_DRI(%)'
    else:
        # Rest data = apply STI 
        return 'BLUP_STI'

scoring_data['Metric_Used'] = scoring_data['Trait'].apply(determine_metric)

# 4. Z-score calculation
z_scores = []

for trait in selected_traits:
    trait_data = scoring_data[scoring_data['Trait'] == trait].copy()
    metric = trait_data['Metric_Used'].iloc[0]
    
    values = trait_data[metric].values
    
    # Z-score standardize
    z = stats.zscore(values, nan_policy='omit')
    
    # Direction Alignment
    if metric == 'BLUP_DRI(%)':
        z = z * -1      # since smaller DRI indicates excellent tolerance
        
    trait_data['Standardized_Z_Score'] = z
    z_scores.append(trait_data)

scoring_data_final = pd.concat(z_scores)

# Calculating composite tolerance score (CTS) per genotype using calculated z-score
genotype_scores = scoring_data_final.groupby('Genotype')['Standardized_Z_Score'].mean().reset_index()
genotype_scores.rename(columns={'Standardized_Z_Score': 'Composite_Tolerance_Score(CTS)'}, inplace=True)

# Ranking
genotype_scores = genotype_scores.sort_values(by='Composite_Tolerance_Score(CTS)', ascending=False).reset_index(drop=True)
genotype_scores['Rank'] = genotype_scores.index + 1


pivot_details = scoring_data_final.pivot(index='Genotype', columns='Trait', values='Standardized_Z_Score')

save_path_phase5 = os.path.join(base_dir, 'Phase5_Adaptive_Genotype_Ranking.csv')
genotype_scores.to_csv(save_path_phase5, index=False, encoding='utf-8-sig')

print("-" * 50)
print("Adaptive Scoring complete.")
print("\ngenotype Rank")
print(genotype_scores.to_string(index=False))

In [ ]:
#######################################
#####  Phase 5.2. Scoring traits  #####
#######################################

import pandas as pd
import matplotlib.pyplot as plt
import os

base_dir = './'
phase4_path = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')

print("Loading data..")
df_p4 = pd.read_csv(phase4_path)

# Set strict trait selection Cut-off (Treatment_p-value < 0.05, GxE variance proportion >= 15%)
P_VALUE_THRESHOLD = 0.05
GXE_THRESHOLD = 15.0

# Satisfied trait filtering
selected_traits_df = df_p4[
    (df_p4['Treatment_p_value'] < P_VALUE_THRESHOLD) & 
    (df_p4['Var_GxE(%)'] >= GXE_THRESHOLD)
].copy()

# Trait scoring and ranking via Var_GxE(%)
selected_traits_df = selected_traits_df.sort_values(by='Var_GxE(%)', ascending=False).reset_index(drop=True)
selected_traits_df['Trait_Rank'] = selected_traits_df.index + 1

save_path_traits = os.path.join(base_dir, 'Phase4_Selected_Traits_Ranking.csv')
selected_traits_df.to_csv(save_path_traits, index=False, encoding='utf-8-sig')

print("-" * 50)
print(f"Trait Selection Complete: {len(selected_traits_df)} traits passed the rigorous cut-off.")
print(f"Results saved to: {save_path_traits}")
print("-" * 50)
print("\n[Top Selected Traits Ranking]")
print(selected_traits_df[['Trait_Rank', 'Trait', 'Treatment_p_value', 'Var_GxE(%)', 'Var_Genotype(%)']].head(10).to_string(index=False))




###---------- Visualization with bar stack chart------------###

try:
    plot_df = selected_traits_df.copy()
    plot_df.set_index('Trait', inplace=True)
    plot_df = plot_df[['Var_GxE(%)', 'Var_Genotype(%)', 'Var_Error(%)']]
    
    colors = ['red', 'blue', 'grey']
    
    fig_height = max(8, len(plot_df) * 0.5)
    fig, ax = plt.subplots(figsize=(10, fig_height))
    
    plot_df.plot(kind='barh', stacked=True, color=colors, ax=ax, edgecolor='black')
    
    ax.set_title('Variance Component Proportions of Selected Traits', fontsize=14, fontweight='bold')
    ax.set_xlabel('Proportion of Total Variance (%)', fontsize=12)
    ax.set_ylabel('Traits', fontsize=12)
    ax.legend(['GxE Interaction', 'Genotype (Baseline)', 'Error (Noise)'], 
              loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
    
    plt.gca().invert_yaxis() # 1st rank trait comes upper
    plt.tight_layout()
    
    plot_save_path = os.path.join(base_dir, 'Phase4_Selected_All_Traits_VCP_Plot.png')
    plt.savefig(plot_save_path, dpi=300)
    print(f"\n[Plot saved] Effect structure visualization saved to: {plot_save_path}")

except Exception as e:
    print(f"\nCould not generate plot: {e}")

In [ ]:
###############################################################################
#####  Phase 5.2 (Additional) Scoring more traits for correlation matrix  #####
###############################################################################

import pandas as pd
import matplotlib.pyplot as plt
import os

base_dir = './'
phase4_path = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')

print("Loading data..")
df_p4 = pd.read_csv(phase4_path)

# Set strict trait selection Cut-off (Treatment_p-value < 0.05, GxE variance proportion >= 15%)
P_VALUE_THRESHOLD = 0.05
GXE_THRESHOLD = 10.0

# Satisfied trait filtering
selected_traits_df = df_p4[
    (df_p4['Treatment_p_value'] < P_VALUE_THRESHOLD) & 
    (df_p4['Var_GxE(%)'] >= GXE_THRESHOLD)
].copy()

# Trait scoring and ranking via Var_GxE(%)
selected_traits_df = selected_traits_df.sort_values(by='Var_GxE(%)', ascending=False).reset_index(drop=True)
selected_traits_df['Trait_Rank'] = selected_traits_df.index + 1

save_path_traits = os.path.join(base_dir, 'Phase4.2_Selected_Traits_Ranking_correlation.csv')
selected_traits_df.to_csv(save_path_traits, index=False, encoding='utf-8-sig')

print("-" * 50)
print(f"Trait Selection Complete: {len(selected_traits_df)} traits passed the rigorous cut-off.")
print(f"Results saved to: {save_path_traits}")
print("-" * 50)
print("\n[Top Selected Traits Ranking]")
print(selected_traits_df[['Trait_Rank', 'Trait', 'Treatment_p_value', 'Var_GxE(%)', 'Var_Genotype(%)']].head(10).to_string(index=False))
